# Passing Quality — Exhaustive Feature Search

All 127 combinations of delta TQ features (always with `from_PQ`).
Find the best subset by test R².

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from itertools import combinations
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
mf = df[df["from_position"] == "Midfielder"].copy()

mf["delta_PQ"] = mf["to_Passing quality"] - mf["from_Passing quality"]

TEAM_QUALITIES = ["ATTACK", "ATTACKING_TRANSITION", "CHANCE_CREATION",
                  "DEFENCE", "DEFENSIVE_TRANSITION", "OUTCOME", "PENETRATION"]

for tq in TEAM_QUALITIES:
    mf[f"delta_tq_{tq}"] = mf[f"to_q_{tq}"] - mf[f"from_q_proj_{tq}"]

delta_tq = [f"delta_tq_{tq}" for tq in TEAM_QUALITIES]
mf = mf.dropna(subset=["from_Passing quality", "delta_PQ"] + delta_tq)

train, test = train_test_split(mf, test_size=0.2, random_state=42)
y_train = train["delta_PQ"]
y_test  = test["delta_PQ"]
print(f"Train: {len(train):,}  |  Test: {len(test):,}")

---
## Naive Baseline

In [ ]:
X_tr = sm.add_constant(train[["from_Passing quality"]])
X_te = sm.add_constant(test[["from_Passing quality"]])
naive = sm.OLS(y_train, X_tr).fit()
naive_pred = naive.predict(X_te)
naive_r2 = r2_score(y_test, naive_pred)
naive_mae = mean_absolute_error(y_test, naive_pred)
print(f"Naive — R²: {naive_r2:.4f}  |  MAE: {naive_mae:.4f}")

---
## Exhaustive Search (127 combinations)

In [ ]:
results = []

for k in range(1, len(delta_tq) + 1):
    for combo in combinations(delta_tq, k):
        feat = ["from_Passing quality"] + list(combo)
        X_tr = sm.add_constant(train[feat])
        X_te = sm.add_constant(test[feat])
        model = sm.OLS(y_train, X_tr).fit()
        pred = model.predict(X_te)
        results.append({
            "n_deltas": k,
            "deltas": ", ".join(c.replace("delta_tq_", "") for c in combo),
            "R2_test": r2_score(y_test, pred),
            "MAE_test": mean_absolute_error(y_test, pred),
            "R2_adj_train": model.rsquared_adj,
            "BIC": model.bic,
        })

results_df = pd.DataFrame(results).sort_values('R2_test', ascending=False)
print(f"Total combinations: {len(results_df)}")

---
## Top 10 Combinations by Test R²

In [ ]:
print(f"Naive baseline R²: {naive_r2:.4f}")
print()
print(results_df.head(10).to_string(index=False, float_format="{:.4f}".format))

---
## Best Combination per Number of Deltas

In [ ]:
best_per_k = results_df.loc[results_df.groupby('n_deltas')['R2_test'].idxmax()]
best_per_k = best_per_k.sort_values('n_deltas')
print(f"Naive baseline R²: {naive_r2:.4f}")
print()
print(best_per_k.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# All combos as grey dots
ax.scatter(results_df['n_deltas'], results_df['R2_test'], alpha=0.3, s=20, color='grey', label='All combos')

# Best per k as blue
ax.scatter(best_per_k['n_deltas'], best_per_k['R2_test'], s=80, color='#2196F3', zorder=5, label='Best per k')

# Naive baseline
ax.axhline(naive_r2, color='red', linestyle='--', linewidth=1, label=f'Naive (R²={naive_r2:.4f})')

ax.set_xlabel('Number of delta TQ features')
ax.set_ylabel('Test R²')
ax.set_title('Test R² by number of delta TQ features (all 127 combos)')
ax.set_xticks(range(1, 8))
ax.legend()
plt.tight_layout()
plt.show()

---
## Winner — Full OLS Summary

In [ ]:
# Re-run the best combo
best = results_df.iloc[0]
print(f"Best combo: {best["deltas"]}")
print(f"R² test: {best["R2_test"]:.4f}  |  MAE: {best["MAE_test"]:.4f}")
print()

best_deltas = [f"delta_tq_{d.strip()}" for d in best["deltas"].split(",")]
best_feat = ["from_Passing quality"] + best_deltas
X_tr = sm.add_constant(train[best_feat])
X_te = sm.add_constant(test[best_feat])
best_model = sm.OLS(y_train, X_tr).fit()
print(best_model.summary())